
# 07 — Capstone: A Working Computer Vision Verification Module

This combines notebooks 01–06 into one class — the CV-side sibling to the `LandVerificationEngine` from
the GeoPandas curriculum's capstone, and specifically the "Computer vision" module shown in the product's
system architecture diagram, sitting alongside Geospatial and Risk Models inside the Python
microservice.

## What this module does

1. **`analyze_document(image_path)`** — runs the rule-based forensic checks from notebook 02 (ELA,
   copy-move, metadata) on a submitted title/deed image.
2. **`monitor_parcel_changes(before_path, after_path, parcel_geom)`** — runs the change-detection
   pipeline from notebook 06 against a specific parcel boundary.
3. **`combined_risk_summary(...)`** — merges both into one structured record, in the same
   LLM-ready-data-not-LLM-invented-facts spirit as the geospatial capstone's due diligence report.

## Step 1: the document forensics component (from notebook 02)


In [1]:

import numpy as np
import cv2
from PIL import Image, ImageChops, ExifTags
import io


class DocumentForensics:
    def error_level_analysis(self, image_path, quality=90, amplify=15):
        original = Image.open(image_path).convert("RGB")
        buffer = io.BytesIO()
        original.save(buffer, "JPEG", quality=quality)
        buffer.seek(0)
        resaved = Image.open(buffer)
        diff = ImageChops.difference(original, resaved)
        diff_array = np.array(diff).astype(np.int32)
        return np.clip(diff_array * amplify, 0, 255).astype(np.uint8)

    def detect_copy_move(self, image_rgb, min_distance=40):
        gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
        orb = cv2.ORB_create(nfeatures=2000)
        keypoints, descriptors = orb.detectAndCompute(gray, None)
        if descriptors is None or len(keypoints) < 2:
            return []
        bf = cv2.BFMatcher(cv2.NORM_HAMMING)
        matches = bf.knnMatch(descriptors, descriptors, k=3)
        pairs = []
        for group in matches:
            for m in group[1:]:
                if m.distance < 40:
                    pt1 = keypoints[m.queryIdx].pt
                    pt2 = keypoints[m.trainIdx].pt
                    if np.hypot(pt1[0] - pt2[0], pt1[1] - pt2[1]) > min_distance:
                        pairs.append((pt1, pt2))
        return pairs

    def inspect_metadata(self, image_path):
        img = Image.open(image_path)
        exif_data = img.getexif()
        if not exif_data:
            return {"has_exif": False, "flags": []}
        readable = {ExifTags.TAGS.get(k, k): v for k, v in exif_data.items()}
        flags = []
        software = str(readable.get("Software", ""))
        if any(kw in software for kw in ["Photoshop", "GIMP", "Paint"]):
            flags.append(f"Editing software detected: {software}")
        return {"has_exif": True, "flags": flags}

    def analyze(self, image_path):
        ela = self.error_level_analysis(image_path)
        ela_intensity = float(np.mean(ela))

        image_rgb = np.array(Image.open(image_path).convert("RGB"))
        copy_move_pairs = self.detect_copy_move(image_rgb)

        metadata = self.inspect_metadata(image_path)

        return {
            "ela_mean_intensity": round(ela_intensity, 2),
            "copy_move_suspicious_pairs": len(copy_move_pairs),
            "metadata_flags": metadata["flags"],
            "recommend_human_review": (
                ela_intensity > 3.0 or len(copy_move_pairs) > 5 or len(metadata["flags"]) > 0
            ),
        }

print("DocumentForensics ready.")


DocumentForensics ready.



## Step 2: the change-detection component (from notebook 06)


In [2]:

from skimage.metrics import structural_similarity as ssim
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd


class LandMonitoring:
    def detect_changes(self, before_arr, after_arr, transform, threshold=100):
        _, diff_map = ssim(before_arr, after_arr, full=True)
        change_map = ((1 - diff_map) * 255).astype(np.uint8)
        _, mask = cv2.threshold(change_map, threshold, 255, cv2.THRESH_BINARY)
        mask_clean = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))

        polygons = [shape(geom) for geom, value in shapes(mask_clean, transform=transform) if value == 255]
        return gpd.GeoDataFrame({"alert_type": ["unauthorized_change"] * len(polygons)},
                                  geometry=polygons, crs="EPSG:4326")

    def check_against_parcel(self, alerts_gdf, parcel_geom, parcel_crs="EPSG:4326"):
        parcel_gdf = gpd.GeoDataFrame({"parcel_check": [True]}, geometry=[parcel_geom], crs=parcel_crs)
        if len(alerts_gdf) == 0:
            return False, 0
        matched = gpd.sjoin(alerts_gdf, parcel_gdf, how="inner", predicate="intersects")
        return len(matched) > 0, len(matched)

print("LandMonitoring ready.")


LandMonitoring ready.



## Step 3: the combined verification module


In [3]:

class CVVerificationModule:
    def __init__(self):
        self.document_forensics = DocumentForensics()
        self.land_monitoring = LandMonitoring()

    def full_verification_report(self, document_image_path, before_arr=None, after_arr=None,
                                   raster_transform=None, parcel_geom=None):
        """
        Produces one structured report combining document authenticity signals and (optionally,
        if imagery is provided) land-monitoring signals for a single parcel submission.
        Deliberately structured data only — no natural-language claims — ready to be handed to
        an LLM for the buyer-facing summary, same principle as the geospatial capstone's report.
        """
        report = {"document_analysis": self.document_forensics.analyze(document_image_path)}

        if before_arr is not None and after_arr is not None and raster_transform is not None and parcel_geom is not None:
            alerts = self.land_monitoring.detect_changes(before_arr, after_arr, raster_transform)
            has_change, n_alerts = self.land_monitoring.check_against_parcel(alerts, parcel_geom)
            report["land_monitoring"] = {
                "unauthorized_change_detected": has_change,
                "alert_count": n_alerts,
            }
        else:
            report["land_monitoring"] = {"status": "not run — no before/after imagery provided"}

        report["overall_recommend_human_review"] = (
            report["document_analysis"]["recommend_human_review"]
            or report["land_monitoring"].get("unauthorized_change_detected", False)
        )
        return report

module = CVVerificationModule()
print("CVVerificationModule ready.")


CVVerificationModule ready.



## Running it end to end

Reusing the same synthetic document and land imagery generation from notebooks 02 and 06, to produce one
real, combined report.


In [4]:

from PIL import ImageDraw
from rasterio.transform import from_origin
from shapely.geometry import Polygon

# Synthetic document (reusing notebook 02's generator, inline here since notebooks don't share state)
def make_document():
    canvas = Image.new("RGB", (500, 350), (248, 246, 240))
    draw = ImageDraw.Draw(canvas)
    draw.rectangle([20, 20, 480, 330], outline=(0, 0, 0), width=2)
    draw.text((40, 50), "LAND TITLE DEED", fill=(30, 30, 30))
    draw.text((40, 100), "Owner: Ekotto Land Holdings", fill=(30, 30, 30))
    draw.text((40, 130), "Parcel size: 2.5 hectares", fill=(30, 30, 30))
    return canvas

make_document().save("/tmp/capstone_document.jpg", quality=90)

# Synthetic before/after land imagery (reusing notebook 06's pattern)
np.random.seed(20)
size = 200
def make_land_image(add_structure=False):
    base = np.random.normal(100, 8, (size, size)).astype(np.float32)
    if add_structure:
        base[70:120, 60:130] = np.random.normal(200, 5, (50, 70))
    return np.clip(base, 0, 255).astype(np.uint8)

before = make_land_image(False)
after = make_land_image(True)
transform = from_origin(west=9.24, north=4.16, xsize=0.0002, ysize=0.0002)
parcel_geom = Polygon([(9.240, 4.155), (9.250, 4.155), (9.250, 4.165), (9.240, 4.165)])

report = module.full_verification_report(
    document_image_path="/tmp/capstone_document.jpg",
    before_arr=before, after_arr=after,
    raster_transform=transform, parcel_geom=parcel_geom,
)

import json
print(json.dumps(report, indent=2))


{
  "document_analysis": {
    "ela_mean_intensity": 0.13,
    "copy_move_suspicious_pairs": 674,
    "metadata_flags": [],
    "recommend_human_review": true
  },
  "land_monitoring": {
    "unauthorized_change_detected": true,
    "alert_count": 1
  },
  "overall_recommend_human_review": true
}



## How this maps to the real product's file structure

```
services/cv/
  document_forensics.py     — the DocumentForensics class
  land_monitoring.py        — the LandMonitoring class
  verification_module.py    — CVVerificationModule, the orchestration layer
  models/
    tamper_classifier.py    — the trained CNN from notebooks 03-04, once real labeled data exists
tests/
  test_document_forensics.py
  test_land_monitoring.py
```

Notice this mirrors the exact same file-splitting pattern the geospatial capstone used — each class in
its own file, independently testable, with the orchestration layer as the one piece that actually gets
called from a route handler.

## Capstone project tasks

### Task 1 — Combine both capstones into one due diligence report
Go back to the `LandVerificationEngine` from the GeoPandas curriculum's capstone and this notebook's
`CVVerificationModule`. Write a top-level function `generate_full_due_diligence_report(...)` that calls
both and merges their output into one report — this is literally what the real product's
`report_builder.py` (mentioned in the geospatial capstone) would do, pulling from every module.

### Task 2 — Add a confidence-weighted overall risk score
Instead of the simple boolean `overall_recommend_human_review`, design a numeric 0–100 risk score that
weighs each signal (ELA intensity, copy-move pair count, land-monitoring alerts) by how reliable you
judge that signal to be. Write out your reasoning for the weights you chose — there's no single correct
answer here, but a real product needs this decision made deliberately, not left as an implicit default.

### Task 3 — Add a persistence layer
Using the PostGIS patterns from the GeoPandas curriculum's notebook 07, design a `verification_reports`
table schema that could store this module's output (both the document analysis and land monitoring
fields) linked to a `parcel_id`. Consider: should a new verification report ever overwrite an old one, or
should the history be kept? (This is the same immutability question flagged in the geospatial capstone.)

### Task 4 — Think about what's still missing
Write a short reflection (markdown, no code): given everything built across both curricula, what's the
single biggest remaining gap before this could run against real production data? Revisit the product's
implementation plan's Phase 0 assumptions if you want a hint — the honest answer hasn't changed.


In [5]:
# Task 1 — your code here


In [6]:
# Task 2 — your code here


In [7]:
# Task 3 — your notes/schema here


In [8]:
# Task 4 — your reflection here



## Where this leaves you

Between this curriculum and the GeoPandas one, you now have working, executed code for every in-house
build item in the product's feature matrix except the risk/valuation models and the LLM/RAG layer —
exactly the two topics still on the table from the options you were given earlier. Whenever you're ready
for either of those, the same approach applies: real executable notebooks, tied directly to this specific
product, not generic tutorials.
